# 02 — Smoke Harness

Reads the most recent smoke run's `events.jsonl`, asserts T0/T1/T6 are present
for every task, and renders an E2E latency histogram.

This notebook is the Phase 0 reporting gate: it must execute cleanly via
`papermill notebooks/02_smoke_harness.ipynb /dev/null`.

## Parameters

In [ ]:
# Injected by Papermill
run_id = None           # Specific run_id to load; None → use most recent
results_path = None     # Base results dir; None → auto-detect
expected_tasks = 100    # Number of tasks expected in the smoke run

## 1. Setup

In [ ]:
import os
from pathlib import Path

if results_path is not None:
    base = Path(results_path)
else:
    base = Path(os.environ.get('BENCH_RESULTS_PATH', Path.cwd() / 'results'))

raw_dir = base / 'raw'
print(f"Results base:  {base}")
print(f"Raw runs dir:  {raw_dir}")
assert raw_dir.exists(), f"Raw results dir does not exist: {raw_dir}"

## 2. Find run

In [ ]:
import json

if run_id is not None:
    target_dir = raw_dir / run_id
else:
    # Use most recent run directory (sorted by name = timestamp prefix)
    run_dirs = sorted(raw_dir.iterdir(), reverse=True)
    assert run_dirs, f"No run directories found under {raw_dir}"
    target_dir = run_dirs[0]
    run_id = target_dir.name

print(f"Loading run: {run_id}")
print(f"Run dir:     {target_dir}")
assert target_dir.exists(), f"Run directory not found: {target_dir}"

# Load run.json if present
run_json_path = target_dir / 'run.json'
if run_json_path.exists():
    with open(run_json_path) as f:
        run_spec = json.load(f)
    print(f"RunSpec broker={run_spec.get('spec', {}).get('broker')}  "
          f"task={run_spec.get('spec', {}).get('task')}")

## 3. Load events

In [ ]:
events_path = target_dir / 'events.jsonl'
assert events_path.exists(), f"events.jsonl not found at {events_path}"

events = []
with open(events_path) as f:
    for line in f:
        line = line.strip()
        if line:
            events.append(json.loads(line))

print(f"Loaded {len(events)} event lines")
print("Sample event:", events[0] if events else '(none)')

## 4. Assertions

In [ ]:
# Merge events by task_id
by_task = {}
for ev in events:
    tid = ev.get('task_id', '')
    if tid not in by_task:
        by_task[tid] = {}
    by_task[tid].update(ev)

n_tasks = len(by_task)
print(f"Unique tasks: {n_tasks}  (expected {expected_tasks})")
assert n_tasks == expected_tasks, (
    f"Expected {expected_tasks} tasks, found {n_tasks}"
)

missing_t0 = [tid for tid, ev in by_task.items() if 'T0' not in ev]
missing_t1 = [tid for tid, ev in by_task.items() if 'T1' not in ev]
missing_t6 = [tid for tid, ev in by_task.items() if 'T6' not in ev]

assert not missing_t0, f"Tasks missing T0: {missing_t0[:5]}"
assert not missing_t1, f"Tasks missing T1: {missing_t1[:5]}"
assert not missing_t6, f"Tasks missing T6: {missing_t6[:5]}"

print("All tasks have T0, T1, T6  ✓")

## 5. E2E latency statistics

In [ ]:
import pandas as pd

rows = []
for tid, ev in by_task.items():
    t0 = ev.get('T0', 0)
    t6 = ev.get('T6', t0)
    e2e_ms = (t6 - t0) / 1_000_000.0
    rows.append({'task_id': tid, 't0_ns': t0, 't6_ns': t6, 'e2e_ms': e2e_ms})

df = pd.DataFrame(rows)

stats = df['e2e_ms'].describe(percentiles=[0.50, 0.95, 0.99])
print(stats.to_string())

p95 = df['e2e_ms'].quantile(0.95)
assert p95 >= 0, "p95 must be non-negative"
print(f"\np95 e2e_ms = {p95:.3f} ms")

## 6. E2E latency histogram

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['e2e_ms'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(df['e2e_ms'].quantile(0.50), color='orange', linestyle='--', label='p50')
ax.axvline(df['e2e_ms'].quantile(0.95), color='red',    linestyle='--', label='p95')
ax.set_xlabel('E2E latency (ms)')
ax.set_ylabel('Task count')
ax.set_title(f'Smoke run E2E latency — {n_tasks} tasks  (run_id: {run_id})')
ax.legend()
plt.tight_layout()

# Save to results dir for later inspection
reports_dir = base / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
fig_path = reports_dir / f'{run_id}_smoke_histogram.png'
fig.savefig(fig_path, dpi=120)
print(f"Histogram saved to {fig_path}")
plt.show()
plt.close(fig)

## Summary

All Phase 0 assertions passed:
- 100 tasks recorded in events.jsonl
- T0, T1, T6 present for every task
- p95 E2E latency is finite and non-negative